# CatalogIQ Dataset Profile

This notebook profiles the supplied training and target CSV files before cleaning or modeling.

It checks:

- row and column counts
- column names
- data types
- missing values
- example records
- duplicate rows
- malformed / shifted CSV rows
- training target labels and class counts

The final cell writes the required report to:

`docs/research/data_profile.md`


In [ ]:
from pathlib import Path
import csv
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

ROOT = Path.cwd().parents[0]
print(ROOT)

TRAIN_PATH = Path(ROOT/"data/provided/Selfcare_Training_data (1).csv")
TARGET_PATH = Path(ROOT/"data/provided/Selfcare_Target_data (1).csv")

REPORT_PATH = Path(ROOT/"docs/findings/notebook_results.md")

TARGET_COLUMNS = [
    "Mnfr",
    "Brand",
    "Platform",
    "Segment",
    "Sub-Segment",
    "TargetAgeGroup",
]


c:\Users\sebas\PycharmProjects\Git\Critical-Materials-Recovery


## 1. Load the datasets

In [28]:
train = pd.read_csv(TRAIN_PATH)
target = pd.read_csv(TARGET_PATH)

print("Training shape:", train.shape)
print("Target shape:  ", target.shape)


C:\Users\sebas\AppData\Local\Temp\ipykernel_6476\4184922433.py:1: DtypeWarning: Columns (10: Sun1, 11: Sun2, 12: Sun3, 13: Sun4, 14: Sun5, 15: Notes) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv(TRAIN_PATH)


Training shape: (84552, 32)
Target shape:   (28082, 32)


C:\Users\sebas\AppData\Local\Temp\ipykernel_6476\4184922433.py:2: DtypeWarning: Columns (11: Sun2, 12: Sun3, 13: Sun4, 14: Sun5, 15: Notes) have mixed types. Specify dtype option on import or set low_memory=False.
  target = pd.read_csv(TARGET_PATH)


## 2. Row counts and column names

In [9]:
print("TRAINING DATA")
print("Rows:", len(train))
print("Columns:", len(train.columns))
print(train.columns.tolist())

print("\nTARGET DATA")
print("Rows:", len(target))
print("Columns:", len(target.columns))
print(target.columns.tolist())


TRAINING DATA
Rows: 84552
Columns: 32
['Exclude', 'JoiningKey', 'Retailer', 'Category', 'Mnfr', 'Brand', 'Platform', 'Segment', 'Sub-Segment', 'TargetAgeGroup', 'Sun1', 'Sun2', 'Sun3', 'Sun4', 'Sun5', 'Notes', 'ProductCategory', 'ProductBrand', 'ProductName', 'ProductRating', 'ProductReviewsCount', 'XRatXRev', 'ReviewsCount', 'Sku', 'Upc', 'ProductModelNumber', 'ProductUrl', 'ProductImageUrl', 'MDM_InsertDateTime', 'MDM_Id', 'ProductDescription', 'ProductContents']

TARGET DATA
Rows: 28082
Columns: 32
['Exclude', 'JoiningKey', 'Retailer', 'Category', 'Mnfr', 'Brand', 'Platform', 'Segment', 'Sub-Segment', 'TargetAgeGroup', 'Sun1', 'Sun2', 'Sun3', 'Sun4', 'Sun5', 'Notes', 'ProductCategory', 'ProductBrand', 'ProductName', 'ProductRating', 'ProductReviewsCount', 'XRatXRev', 'ReviewsCount', 'Sku', 'Upc', 'ProductModelNumber', 'ProductUrl', 'ProductImageUrl', 'MDM_InsertDateTime', 'MDM_Id', 'ProductDescription', 'ProductContents']


## 3. Data types

In [10]:
print("TRAINING DATA TYPES")
display(train.dtypes.rename("dtype").to_frame())

print("TARGET DATA TYPES")
display(target.dtypes.rename("dtype").to_frame())


TRAINING DATA TYPES


,dtype
Exclude,str
JoiningKey,str
Retailer,str
Category,str
Mnfr,str
Brand,str
Platform,str
Segment,str
Sub-Segment,str
TargetAgeGroup,str


TARGET DATA TYPES


,dtype
Exclude,str
JoiningKey,str
Retailer,str
Category,float64
Mnfr,float64
Brand,float64
Platform,float64
Segment,float64
Sub-Segment,float64
TargetAgeGroup,float64


### Training Data Type Notes

Most columns in the training dataset were loaded as strings.

Some columns appear to represent numeric or date values but are currently stored as strings, including:

* `ProductRating`
* `ProductReviewsCount`
* `XRatXRev`
* `ReviewsCount`
* `MDM_InsertDateTime`

This does not necessarily mean the data is incorrect. These columns may contain formatting, mixed values, missing-value markers, or other content that caused pandas to interpret them as strings.

No type conversion will be performed during profiling. These columns should be inspected further during the cleaning stage if their intended data type needs clarification.

### Target Data Type Notes

Most product-information columns in the target dataset were loaded as strings, consistent with the training dataset.

The following classification columns were loaded as `float64`:

* `Category`
* `Mnfr`
* `Brand`
* `Platform`
* `Segment`
* `Sub-Segment`
* `TargetAgeGroup`

These columns may be entirely or mostly missing in the target dataset. Pandas commonly assigns `float64` to columns containing only missing (`NaN`) values.

This should be verified through the missing-value analysis before drawing conclusions about the intended schema.

No data types will be changed during profiling.



In [16]:
numeric_candidates = [
    "ProductRating",
    "ProductReviewsCount",
    "XRatXRev",
    "ReviewsCount",
    "MDM_Id",
]

for col in numeric_candidates:
    print(f"\n--- {col} ---")

    converted = pd.to_numeric(train[col], errors="coerce")

    # Values that existed originally but could not be converted to numbers
    bad_values = train.loc[
        train[col].notna() & converted.isna(),
        col
    ]

    print("Could not convert:", len(bad_values))
    print("Examples:")
    print(bad_values.drop_duplicates().head(20).tolist())


--- ProductRating ---
Could not convert: 462
Examples:
[' with 4 Heat Levels & 2H Auto-off', ' Shoulder', ' 9"" &11"" ] Hot and Cold Reusable Ice Bag', 'Machine Washable Red"', ' Moist Heat Pack Gifts for Women & Girls', ' Wisdom Teeth', ' 2 Hours Auto Shut Off Available', ' Auto Off and MEM Function"', 'Chrictmas Gifts for Women/Men/Dad/Mom"', ' Auto Shut-Off', ' 4 Hours Auto Shut Off', ' Fast Heated with 10 Temperature Settings', ' Hot & Cold Therapy and Pain Relief for Knee Leg Injury', ' Green"', ' Neck and Shoulder', ' Auto Shut Off', ' Black)"', ' Mask and Accessories', ' Back Pain Relief', ' Face']

--- ProductReviewsCount ---
Could not convert: 305
Examples:
[' Dark Blue"', ' Knee', 'Relief Heat Pack Sports Injury Reusable First Aid for Knee Head Leg(Deep Blue Snowflake)"', ' Pink"', ' Fever', 'Machine Washable"', ' and Machine Washable Foot Warmer Under Desk', ' Electric Heating Pad with 5 Heat Settings', ' Auto Shut Off', ' 4-Pack (6""/9""/11"")', ' 5-Pack (6""/9""/11"") No-

### Possible Shifted or Misaligned Records

Several columns contain values that do not match their expected meaning.

Examples include:

* `ProductRating` containing product-description text
* `ProductReviewsCount` containing product-name or description fragments
* `XRatXRev` and `ReviewsCount` containing descriptive text rather than numeric values
* `MDM_Id` containing image URLs, product URLs, and product identifiers such as Amazon ASINs

This pattern suggests that some records may be shifted across columns or were parsed incorrectly during creation/export of the CSV.

In [15]:
columns_to_check = [
    "Category",
    "Mnfr",
    "Brand",
    "Platform",
    "Segment",
    "Sub-Segment",
    "TargetAgeGroup",
]

for col in columns_to_check:
    print(
        col,
        "| training missing:", train[col].isna().sum(),
        "| testing missing:", target[col].isna().sum(),
        "| testing rows:", len(target)
    )

Category | training missing: 1 | testing missing: 28082 | testing rows: 28082
Mnfr | training missing: 5698 | testing missing: 28082 | testing rows: 28082
Brand | training missing: 322 | testing missing: 28082 | testing rows: 28082
Platform | training missing: 83447 | testing missing: 28082 | testing rows: 28082
Segment | training missing: 4789 | testing missing: 28082 | testing rows: 28082
Sub-Segment | training missing: 7515 | testing missing: 28082 | testing rows: 28082
TargetAgeGroup | training missing: 7077 | testing missing: 28082 | testing rows: 28082


### Classification Column Missingness

All classification columns are completely missing in the target/testing dataset.

* `Category`: 28,082 of 28,082 missing
* `Mnfr`: 28,082 of 28,082 missing
* `Brand`: 28,082 of 28,082 missing
* `Platform`: 28,082 of 28,082 missing
* `Segment`: 28,082 of 28,082 missing
* `Sub-Segment`: 28,082 of 28,082 missing
* `TargetAgeGroup`: 28,082 of 28,082 missing

This explains why pandas loads these target columns as `float64`: columns containing only missing values default to a numeric NaN-compatible type.

The training dataset also contains missing labels:

* `Category`: 1 missing
* `Mnfr`: 5,698 missing
* `Brand`: 322 missing
* `Platform`: 83,447 missing
* `Segment`: 4,789 missing
* `Sub-Segment`: 7,515 missing
* `TargetAgeGroup`: 7,077 missing

`Platform` has substantially more missing training labels than the other classification fields and should be examined further before modeling.

An additional point requiring clarification is that `Category` is fully blank in the target dataset even though the current project scope identifies the other six fields as prediction targets.

In [17]:
print("Total training rows:", len(train))
print("Platform labeled:", train["Platform"].notna().sum())
print("Platform missing:", train["Platform"].isna().sum())
print("Platform missing %:", train["Platform"].isna().mean() * 100)

print("\nUnique Platform labels:")
print(train["Platform"].value_counts(dropna=False))

Total training rows: 84552
Platform labeled: 1105
Platform missing: 83447
Platform missing %: 98.69311193111932

Unique Platform labels:
Platform
NaN                                                                                                     83447
Extra Strength                                                                                             84
Children                                                                                                   72
Pain + Fever                                                                                               51
Oral Suspension                                                                                            49
                                                                                                        ...  
12 Hour Relief                                                                                              1
 orange juice or a smoothie daily; unflavored GRASS-FED: Super Collagen Peptides pow

In [18]:
pd.crosstab(
    train["Category"],
    train["Platform"].isna(),
    margins=True
)

Platform,False,True,All
Category,,,
1-month supply; simply dissolve 2 scoops of Super Collagen Peptides powder into a glass of water,1,0,1
an essential antioxidant,2,0,2
Self Care,1102,83446,84548
All,1105,83446,84551


In [19]:
pd.crosstab(
    train["Segment"],
    train["Platform"].isna(),
    margins=True
)

Platform,False,True,All
Segment,,,
IGEN Non-GMO tested.* No soy wheat,2,0,2
Keto certified,1,0,1
Allergy,225,2512,2737
CCFS,170,4097,4267
Digestive Health,135,6967,7102
External Analgesics,50,6019,6069
Internal Analgesics,420,4388,4808
Lifestyle CHC,0,4551,4551
Other Self Care,54,1081,1135


### Platform Label Availability

`Platform` is extremely sparsely labeled in the training dataset.

* Total training records: 84,552
* Records with a Platform label: 1,105
* Records without a Platform label: 83,447
* Missing Platform labels: 98.69%
* Observed Platform values: 64 including missing values

Some observed Platform values also appear malformed or misaligned. For example, at least one Platform value contains product-description text rather than an apparent classification label.

The dataset therefore presents two separate concerns:

1. Very limited supervised Platform labels.
2. Possible malformed or column-shifted records among the labeled examples.

### Suggested Approach for `Platform`

`Platform` is extremely sparsely labeled in the training data:

* 84,552 total training rows
* 1,105 rows with a non-missing `Platform`
* 83,447 rows missing `Platform`
* 98.69% missing

There also appear to be some malformed or shifted records among the non-missing Platform values, so the usable labeled count may be even lower after cleaning.

#### Suggested approach

For the first modeling attempt, treat missing `Platform` values as **unlabeled**, rather than assuming they mean `"None"`, `"Unknown"`, or `"Not Applicable"`.

Under this approach:

* Keep `Platform` as a prediction target.
* Train the Platform model only on rows with a valid non-missing Platform label.
* Do not assign a meaning to missing Platform values without evidence.
* Evaluate whether the remaining labeled examples provide enough support for useful predictions.
* If Platform performance is poor, document label sparsity as a data limitation rather than automatically removing the target.

This is only a proposed approach. Team feedback is encouraged, especially if there is a better interpretation of the missing Platform values or a better strategy for handling the limited labeled data.

## 4. Missing values

In [11]:
def missing_summary(df):
    summary = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percent": (df.isna().mean() * 100).round(2)
    })
    return summary.sort_values("missing_count", ascending=False)

train_missing = missing_summary(train)
target_missing = missing_summary(target)

print("TRAINING MISSINGNESS")
display(train_missing)

print("TARGET MISSINGNESS")
display(target_missing)


TRAINING MISSINGNESS


,missing_count,missing_percent
Notes,84551,100.00
Sun4,84550,100.00
Sun2,84550,100.00
Sun3,84550,100.00
Sun5,84550,100.00
Sun1,84549,100.00
Platform,83447,98.69
Exclude,79255,93.74
ProductModelNumber,72565,85.82
Upc,67477,79.81


TARGET MISSINGNESS


,missing_count,missing_percent
Category,28082,100.00
Platform,28082,100.00
Brand,28082,100.00
Mnfr,28082,100.00
Segment,28082,100.00
Sub-Segment,28082,100.00
TargetAgeGroup,28082,100.00
Sun2,28081,100.00
Sun5,28081,100.00
Sun4,28081,100.00


### Missingness Observations

The training and target datasets show very similar missing-value patterns across the product-information fields. This suggests that both datasets likely come from the same or a very similar data-generation process.

Several fields are almost entirely empty in both datasets:

* `Sun1`–`Sun5`
* `Notes`

Other fields also have substantial missingness in both datasets:

* `Exclude`: approximately 93%
* `ProductModelNumber`: approximately 86%
* `Upc`: approximately 80%
* `ProductDescription`: approximately 68%
* `ProductContents`: approximately 60%

In contrast, several potentially useful product fields are mostly populated, including:

* `ProductName`
* `ProductBrand`
* `ProductCategory`
* `Retailer`
* `JoiningKey`

The classification fields are completely missing in the target dataset, which is consistent with their use as prediction targets.

`Platform` is a notable exception in the training data. Unlike the other classification fields, it is missing for 98.69% of training records. This will require separate consideration when the team begins modeling.

In [20]:
sun_cols = ["Sun1", "Sun2", "Sun3", "Sun4", "Sun5"]

for col in sun_cols:
    print(f"\n--- {col} ---")
    print(train[col].dropna().tolist())


--- Sun1 ---
[' IGEN Non-GMO tested;* contains no soy wheat', 'Protein, Vitamin C (as calcium ascorbate), Biotin, Sodium, Hydrolyzed bovine collagen. OTHER INGREDIENTS: Magnesium stearate. May Contain: trace amounts of naturally occurring sulfite residue.', ' and formulated for healthy hair']

--- Sun2 ---
[' lactose', ' beautiful skin']

--- Sun3 ---
[' starch', ' and nail support.* As we age']

--- Sun4 ---
[' corn or artificial flavors"', ' our collagen gets depleted']

--- Sun5 ---
['Serving Size 2 Scoops (20 g). Servings per Container 30. Calories: 70; Protein 17 g; Sodium: 50 mg; Hydrolyzed Bovine Collagen: 20 g. CONTAINS NO: soy, wheat, lactose, starch, corn or artificial flavors. Gluten-Free. May Contain: trace amounts of naturally occurring sulfite residue.', ' which can lead to many common signs of aging. NeoCell Super Collagen + Vitamin C and Biotin supports healthy collagen formation with type 1 and 3 and supports healthy joints.* Hydrolyzed collagen provides some of the a

### Suggested Handling for `Sun1`–`Sun5`

The meaning of `Sun1`–`Sun5` is currently unknown. In the supplied datasets, these columns are nearly 100% missing, and the few populated values appear to contain fragments from malformed or shifted records.

Because the intended purpose of these fields is unknown, permanently removing them from the data pipeline may be premature. A future dataset release could populate these fields with meaningful information.

#### Suggested approach

* Preserve `Sun1`–`Sun5` in the raw and profiled datasets.
* Exclude them from the current modeling/prediction feature set.
* Monitor their non-missing percentage whenever new data is ingested.
* If any `Sun` column becomes more than **1% populated**, flag it for review before the next model run.
* Do not automatically begin using the field as a feature; first determine what it represents and whether its values are valid.

## 5. Example records

In [12]:
print("TRAINING EXAMPLES")
display(train.head(5))

print("TARGET EXAMPLES")
display(target.head(5))


TRAINING EXAMPLES


,Exclude,JoiningKey,Retailer,Category,Mnfr,Brand,Platform,Segment,Sub-Segment,TargetAgeGroup,Sun1,Sun2,Sun3,Sun4,Sun5,Notes,ProductCategory,ProductBrand,ProductName,ProductRating,ProductReviewsCount,XRatXRev,ReviewsCount,Sku,Upc,ProductModelNumber,ProductUrl,ProductImageUrl,MDM_InsertDateTime,MDM_Id,ProductDescription,ProductContents
0,NaN,00014b3db41adfe3911aefbfa88b05af,Amazon,Self Care,All others,UnItemised brand,NaN,External Analgesics,Creams/Gels & Medicated Patches,Adult,NaN,NaN,NaN,NaN,NaN,NaN,Health & Household > Health Care > Over-the-Counter Medication > Pain Reliev...,Cora,Cora Period Menstrual Cramp Relief Kit Period Balm with Soothing Warmth (1 o...,3.9,50,15.6,4,B08QDPHXJF,NaN,NaN,https://www.amazon.com/dp/B08QDPHXJF,https://m.media-amazon.com/images/I/71XzvQ+vftL._AC_SL1500_.jpg | https://m....,45117.83254,1136754,NaN,INTUITIVELY DESIGNED. Soothe cramps where you need it most. The patch adhere...
1,NaN,00015d42f7052a8d5c9d3489d1ac5a63,Amazon,Self Care,All others,UnItemised brand,NaN,Internal Analgesics,Speciality Pain,Adult,NaN,NaN,NaN,NaN,NaN,NaN,Health & Household > Health Care > OTC Medications & Treatments > Pain Relie...,WellnessPartners,UTI Slip D-Mannose Non GMO Organic Source Veggie Capsules 90 Count (500 Mg),4.5,50,9,2,B00528Z60O,NaN,NaN,https://www.amazon.com/dp/B00528Z60O,https://m.media-amazon.com/images/I/71vOnIuiyjL._AC_SL1500_.jpg | https://m....,44936,291986,NaN,NaN
2,NaN,000220380ac4fcd79388ecf4fe26488a,Amazon,Self Care,NaN,UnItemised brand,NaN,"Vitamins, Minerals & Supplements",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Health & Household > Vitamins, Minerals & Supplements > Herbal Supplements >...",Ellie's Best,Ellie's Best Lion’s Mane Mushroom Powder Extract Supplement - USDA Organic -...,3.6,5,7.2,2,B0B3NGHTWD,NaN,NaN,https://www.amazon.com/dp/B0B3NGHTWD,https://m.media-amazon.com/images/I/81m4gRGGVkL._AC_SL1500_.jpg | https://m....,45177.76383,2506345,NaN,100% USDA CERTIFIED ORGANIC Lion's Mane Mushroom Extract. DOUBLE WATER EXTRA...
3,NaN,0003d86e9f1502b501aabd81294cfbc1,Walmart,Self Care,All others,Centrum,NaN,"Vitamins, Minerals & Supplements",Multivitamins,Adult,NaN,NaN,NaN,NaN,NaN,NaN,Health and Medicine>Vitamins and Supplements>Multivitamins>50 Plus Multivita...,Centrum,"Centrum Silver Multivitamin for Adults 50 Plus, Multimineral Supplement, 150 Ct",4.7,46,319.6,68,893202,3.00054E+11,4177-59,https://www.walmart.com/ip/Centrum-Silver-Multivitamin-for-Adults-50-Plus-Mu...,https://i5.walmartimages.com/seo/Centrum-Silver-Multivitamin-for-Adults-50-P...,44936,332923,NaN,NaN
4,NaN,000515b579fa1b8bc359d6c48569ae63,Amazon,Self Care,All others,UnItemised brand,NaN,Internal Analgesics,Speciality Pain,Adult,NaN,NaN,NaN,NaN,NaN,NaN,Health & Household > Health Care > OTC Medications & Treatments > Pain Relie...,Counterpain,Counterpain Muscular Pain Relief Hot Warm Balm 120g.,4.5,9,31.5,7,B00A1246ES,NaN,NaN,https://www.amazon.com/dp/B00A1246ES,https://m.media-amazon.com/images/I/71562Xx9MwS._AC_SL1500_.jpg | https://m....,44936,294779,NaN,NaN


TARGET EXAMPLES


,Exclude,JoiningKey,Retailer,Category,Mnfr,Brand,Platform,Segment,Sub-Segment,TargetAgeGroup,Sun1,Sun2,Sun3,Sun4,Sun5,Notes,ProductCategory,ProductBrand,ProductName,ProductRating,ProductReviewsCount,XRatXRev,ReviewsCount,Sku,Upc,ProductModelNumber,ProductUrl,ProductImageUrl,MDM_InsertDateTime,MDM_Id,ProductDescription,ProductContents
0,NaN,000309b7492984f5f63b75a7fd898738,Amazon,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Health & Household > Vitamins, Minerals & Supplements > Digestive Supplement...",Swanson,Swanson Prebiotic + Probiotic Fiber - Natural Supplement Promoting Digestive...,5,1,5,1,B0BCMTGNY6,NaN,NaN,https://www.amazon.com/dp/B0BCMTGNY6,https://m.media-amazon.com/images/I/71Tyrt2VlrL._AC_SL1200_.jpg | https://m....,45117.83254,1136755,NaN,DIGESTIVE HEALTH Support: Swanson's premium Probiotic + Prebiotic supplement...
1,NaN,0005475457eac31e1b48113d32596657,Amazon,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Health & Household > Oral Care > Oral Pain Relief > Emergency Dental Care,Topicy,"Tooth Repair Kit, Temporary Teeth Replacement Kit for Temporary Restoration ...",4.7,146,47,10,B0C5M7G2LS,NaN,NaN,https://www.amazon.com/dp/B0C5M7G2LS,https://m.media-amazon.com/images/I/71p78eoD5bL._AC_SL1500_.jpg | https://m....,45145.83061,1258179,NaN,[Safe & High Quality] The dental restoration kit uses thermoformed beads mad...
2,NaN,0007449da680ef6267cdad691f882fc9,Amazon,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Health & Household > Vitamins, Minerals & Supplements > Blended Vitamin & Mi...",MICHAEL'S,Michael's Naturopathic Programs Kidney Factors - 120 Vegetarian Tablets - Nu...,4.6,42,41.4,9,B0009RC4SU,NaN,NaN,https://www.amazon.com/dp/B0009RC4SU,https://m.media-amazon.com/images/I/61PffVGNF2L._AC_SL1500_.jpg | https://m....,44936,286907,NaN,NaN
3,NaN,000989c388fee65a67a1b438cdcab1a0,Amazon,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Health & Household > Vitamins, Minerals & Supplements > Herbal Supplements >...",Micro Ingredients,"Sustainably US Grown, Organic Barley Grass Powder, 20 Ounce (1.25 Pounds), R...",4.5,136,63,14,B01MT7T5XX,NaN,NaN,https://www.amazon.com/dp/B01MT7T5XX,https://m.media-amazon.com/images/I/614RPd5mQRL._AC_SL1366_.jpg | https://m....,44936,307895,NaN,NaN
4,NaN,0011ea8dbe27bb276b2f8aec9a1cd142,Amazon,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Health & Household > Vitamins, Minerals & Supplements > Essential Fatty Acid...",Toplux,Toplux Keto Pills Ketosis Diet - Natural Ketosis Using Ketone & Ketogenic Di...,3.5,1100,77,22,B094VKZZ5L,NaN,NaN,https://www.amazon.com/dp/B094VKZZ5L,https://m.media-amazon.com/images/I/81RRYFdp+bL._AC_SL1500_.jpg | https://m....,44967,372476,NaN,NaN


### Additional Schema Observations

Several fields appear to require special handling during the later cleaning and feature-engineering stages.

#### `MDM_InsertDateTime`

Some values appear as decimal numbers such as `45117.83254`.

These may represent Excel-style serial date/time values rather than ordinary text timestamps. The column should be inspected further before conversion because multiple datetime formats may be present.

#### `Upc`

`Upc` appears to contain numeric-looking identifiers, including values displayed in scientific notation such as `3.00054E+11`.

Although UPC values contain digits, they should be treated as identifiers rather than numeric measurements. Numeric storage may alter formatting, including leading zeros or scientific-notation representation.

#### `ProductCategory`

`ProductCategory` appears to contain hierarchical category paths such as:

`Health & Household > Health Care > Over-the-Counter Medication > Pain Relief`

If this hierarchy is consistently structured, it may later be useful to extract individual category levels as separate features.

Before doing so, the number of hierarchy levels and consistency of the delimiter should be profiled.

In [21]:
category_levels = (
    train["ProductCategory"]
    .dropna()
    .astype(str)
    .str.split(">")
    .apply(len)
)

print(category_levels.value_counts().sort_index())

ProductCategory
1        1
2     1090
3    23453
4    38788
5    14012
6     6533
7      644
8        1
9        1
Name: count, dtype: int64


In [23]:
for level_count in sorted(category_levels.unique()):
    print(f"\n--- {level_count} levels ---")

    matching_rows = category_levels[category_levels == level_count].index

    print(
        train.loc[matching_rows, "ProductCategory"]
        .head(5)
        .tolist()
    )


--- 1 levels ---
[' cure']

--- 2 levels ---
['Health > Vitamins & Supplements', 'Health > Vitamins & Supplements', 'Health > Vitamins & Supplements', 'Health > Vitamins & Supplements', 'Health > Vitamins & Supplements']

--- 3 levels ---
['Health & Household > Vitamins, Minerals & Supplements > Blended Vitamin & Mineral Supplements', 'Health & Medicine > Pain & Fever > Hot or Cold Therapy', 'Health & Household > Vitamins, Minerals & Supplements > Blended Vitamin & Mineral Supplements', 'Health & Household > Vitamins, Minerals & Supplements > MSM', 'Health and Medicine > Vitamins and Supplements > Cognitive Health Supplements']

--- 4 levels ---
['Health & Household > Vitamins, Minerals & Supplements > Herbal Supplements > Mushrooms', 'Health and Medicine>Vitamins and Supplements>Multivitamins>50 Plus Multivitamins', 'Health & Household > Health Care > Eye Health > Moisturizing Eye Drops', 'Health & Household > Health Care > OTC Medications & Treatments > Pain Relievers', 'Shop>Medici

In [24]:
for level_count in [1, 8, 9]:
    matching_rows = category_levels[category_levels == level_count].index

    display(
        train.loc[
            matching_rows,
            ["ProductCategory", "ProductName", "Retailer"]
        ]
    )

,ProductCategory,ProductName,Retailer
70020,cure,Serving Size 3 Tablets. Calories: 10; Protein: 3 g; Vitamin C (as calcium as...,19 amino acids nourish the skins dermal layers; and Vitamin C


,ProductCategory,ProductName,Retailer
83929,"Health & Household > Vitamins, Minerals & Supplements > Herbal Supplements >...",Licorice Root Extract – Stomach Relief - Powerful Licorice Supplement - Lung...,Amazon


,ProductCategory,ProductName,Retailer
20527,Shop>Medicines & Treatments>Digestive Health & Nausea>Diarrhea Relief>Home>S...,Intensive Bowel Support Probiotic Capsules,Walgreens


### `ProductCategory` Data Quality Findings

`ProductCategory` generally contains a hierarchical category breadcrumb separated by `>`.

Most records contain between 2 and 7 hierarchy levels and appear structurally valid. Different depths likely reflect different retailer taxonomies or different levels of category specificity.

A small number of abnormal records were identified:

* A 1-level value of `cure` appears in a row where other fields are also visibly shifted or malformed.
* An 8-level record contains the same 4-level hierarchy repeated twice.
* A 9-level record contains a duplicated category path with an additional `Home` element between the repeated sections.

These findings suggest that `ProductCategory` is potentially useful for later feature engineering, but should first be cleaned for:

* malformed or shifted records
* duplicated category paths
* inconsistent spacing around the `>` delimiter

The variable hierarchy depth itself should not be treated as an error.

## 6. Duplicate rows

In [25]:
train_duplicate_count = train.duplicated().sum()
target_duplicate_count = target.duplicated().sum()

print("Training duplicate rows:", train_duplicate_count)
print("Target duplicate rows:  ", target_duplicate_count)

if train_duplicate_count > 0:
    print("\nExample training duplicates:")
    display(train[train.duplicated(keep=False)].head(10))

if target_duplicate_count > 0:
    print("\nExample target duplicates:")
    display(target[target.duplicated(keep=False)].head(10))


Training duplicate rows: 0
Target duplicate rows:   0


## 7. Check for malformed or shifted CSV rows

A common CSV problem is a row having a different number of fields than the header.

Using Python's `csv.reader` is safer than simply counting commas because it respects quoted commas.


In [29]:
def find_bad_csv_rows(path, max_examples=20):
    bad_rows = []

    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)

        header = next(reader)
        expected_fields = len(header)

        for line_number, row in enumerate(reader, start=2):
            if len(row) != expected_fields:
                bad_rows.append({
                    "line_number": line_number,
                    "expected_fields": expected_fields,
                    "actual_fields": len(row),
                    "row_preview": row[:8]
                })

                if len(bad_rows) >= max_examples:
                    break

    return expected_fields, bad_rows


train_expected_fields, train_bad_rows = find_bad_csv_rows(TRAIN_PATH)
target_expected_fields, target_bad_rows = find_bad_csv_rows(TARGET_PATH)

print("Training expected fields per row:", train_expected_fields)
print("Training malformed row examples:", len(train_bad_rows))
display(pd.DataFrame(train_bad_rows))

print("Target expected fields per row:", target_expected_fields)
print("Target malformed row examples:", len(target_bad_rows))
display(pd.DataFrame(target_bad_rows))


Training expected fields per row: 32
Training malformed row examples: 0


""


Target expected fields per row: 32
Target malformed row examples: 0


""


## 8. Look for suspicious rows after loading

Rows with unusually many missing fields can also indicate bad or shifted records.


In [33]:
def suspicious_missing_rows(df, threshold=0.50):
    missing_fraction = df.isna().mean(axis=1)
    return df[missing_fraction >= threshold].copy()

target_features = target.drop(columns=TARGET_COLUMNS)

train_suspicious = suspicious_missing_rows(train)
target_suspicious = suspicious_missing_rows(target_features)

print(f"Training rows with 50% or more fields missing:", len(train_suspicious))
display(train_suspicious.head(10))

print(f"Target rows with 50% or more non-target fields missing:", len(target_suspicious))
display(target_suspicious.head(10))


Training rows with 50% or more fields missing: 90


,Exclude,JoiningKey,Retailer,Category,Mnfr,Brand,Platform,Segment,Sub-Segment,TargetAgeGroup,Sun1,Sun2,Sun3,Sun4,Sun5,Notes,ProductCategory,ProductBrand,ProductName,ProductRating,ProductReviewsCount,XRatXRev,ReviewsCount,Sku,Upc,ProductModelNumber,ProductUrl,ProductImageUrl,MDM_InsertDateTime,MDM_Id,ProductDescription,ProductContents
1865,NaN,0c455549d54c8ab743872f2af7039ab2,Amazon,Self Care,NaN,UnItemised brand,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Health & Household > Health Care > Over-the-Counter Medication > Pain Reliev...,FUJI ORCHARD,de 5,4.8,49,235.2,49,B078YHCG15,NaN,NaN,https://www.amazon.com/dp/B078YHCG15,https://m.media-amazon.com/images/I/41dGVHlqlYL._AC_.jpg | https://m.media-a...,45177.76383,2506504,NaN,NaN
1958,NaN,0ced5bd891417cbe36167b14a3c3364d,Amazon,Self Care,NaN,UnItemised brand,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Health & Household > Health Care > Over-the-Counter Medication > Cough & Col...,Snake Brand,Snake Brand Herbal Throat Spray Chamomile Mint Extract - Rescue Series - 15 ...,4,1,4,1,B0BZYJHXJ6,NaN,NaN,https://www.amazon.com/dp/B0BZYJHXJ6,https://m.media-amazon.com/images/I/71oBBO-OWFL._AC_SL1500_.jpg | https://m....,45177.76383,2506388,NaN,NaN
2846,NaN,1482259d1df468f9601a2c2822f46abd,Amazon,Self Care,NaN,HALLS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Health & Household > Health Care > Over-the-Counter Medication > Cough & Col...,Halls,Halls Mentholyptus Citrus Sugar Free Stick Pack std (Pack of 20),4.6,45,4.6,1,B009JS156W,NaN,NaN,https://www.amazon.com/dp/B009JS156W,https://m.media-amazon.com/images/I/81BieWlvtSL._SL1500_.jpg | https://m.med...,45177.76383,2506644,NaN,NaN
3292,NaN,18d2ff70e5b7c10b459d474a67d97a75,Amazon,Self Care,NaN,UnItemised brand,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Health & Household > Health Care > Over-the-Counter Medication > Digestion &...,LERAMED,PACK OF 3 EACH FLETCHERS CASTORIA CHILD LAX 2.5OZ PT#31074200321,4.2,12,4.2,1,B004LTXLF2,NaN,NaN,https://www.amazon.com/dp/B004LTXLF2,https://m.media-amazon.com/images/I/71GsDcljvNL._AC_SL1500_.jpg | https://m....,45177.76383,2506522,NaN,NaN
5261,Exclude,065ecb35741d124fb483ce13d6ec3b2c,Amazon,Self Care,NaN,Other Brands,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Health & Household > Oral Care > Oral Pain Relief > Emergency Dental Care,VEZE,"Tooth Repair Kit, Temporary Fake Teeth Replacement Kit for Temporary Restora...",4,NaN,4,1,B0C1K8XQVL,NaN,NaN,https://www.amazon.com/dp/B0C1K8XQVL,NaN,45117.83254,1137024,NaN,??[Safe & High Quality] The dental restoration kit uses thermoformed beads m...
5642,NaN,03ce574042f7c54f0d0dc71ba4a50677,Amazon,Self Care,NaN,UnItemised brand,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Health & Household > Health Care > Eye Health,Schwabe,Schwabe Cineraria Maritima Eye Drops Without Alcohol 10ml (Pack of 5),4.5,28,4.5,1,B00TTYFZWG,NaN,NaN,https://www.amazon.com/dp/B00TTYFZWG,https://m.media-amazon.com/images/I/41I+UiCPFPL._AC_.jpg,45177.76383,2507029,NaN,NaN
9550,NaN,4a4b66cac001d4fc80831744d7753437,Amazon,Self Care,NaN,Generic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Health & Household > Health Care > Over-the-Counter Medication > Digestion &...,Generic,"Gastritis Herbal/Tea 4 Oz-113gr. Antinflammatory Gastritis, H. Pylori, Heart...",3.9,3,7.8,2,B0B12SPMDV,NaN,NaN,https://www.amazon.com/dp/B0B12SPMDV,https://m.media-amazon.com/images/I/61yo+6X8gZL._AC_SL1000_.jpg | https://m....,45177.76383,2506817,NaN,NaN
10344,NaN,6d8a937b0d88e59251386d2e7f854ce9,Amazon,Self Care,NaN,Other Brands,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Health & Household > Health Care > Over-the-Counter Medication > Cough & Col...,Pun Intented,Pun Intended Energy Lozenges / Hard Candy - Pack of 10,3.9,138,23.4,6,B0958R2T6S,NaN,NaN,https://www.amazon.com/dp/B0958R2T6S,https://m.media-amazon.com/images/I/51do+CVFBwL._SL1000_.jpg | https://m.med...,45177.76383,2506879,NaN,NaN
10878,Exclude,2a365d617412fbd6dee78e1c25281817,Kroger,Self Care,All others,Nature Made,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Home > Health Care > Specialty Tre

Target rows with 50% or more non-target fields missing: 65


,Exclude,JoiningKey,Retailer,Category,Sun1,Sun2,Sun3,Sun4,Sun5,Notes,ProductCategory,ProductBrand,ProductName,ProductRating,ProductReviewsCount,XRatXRev,ReviewsCount,Sku,Upc,ProductModelNumber,ProductUrl,ProductImageUrl,MDM_InsertDateTime,MDM_Id,ProductDescription,ProductContents
457,NaN,088f959cc3fe5c1467d218a41ad37760,Kroger,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Home > Health Care > Vitamins & Supplements,One A Day,NaN,4.2,68,12.6,3,items_457-91534,NaN,NaN,https://www.instacart.com/store/kroger/products/91534,NaN,45055.70023,584761,NaN,NaN
766,NaN,0f97cc383f4a8445866c0b9550e42f7e,Kroger,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Home > Health Care > Vitamins & Supplements,Nature's Way,NaN,4.8,41,168,35,items_457-3392966,NaN,NaN,https://www.instacart.com/store/kroger/products/3392966,NaN,45058.43501,680617,NaN,NaN
1662,NaN,66764bcf30b80cdff96473f991781933,Kroger,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Home > Health Care > Cold, Flu & Allergy",Halls,NaN,4.92,NaN,59.04,12,items_457-25131379,NaN,NaN,https://www.instacart.com/store/kroger/products/25131379,https://d2lnr5mha7bycj.cloudfront.net/product-image/file/large_ff108e28-1eb5...,45058.43501,680683,NaN,NaN
1700,HYDROLYZED COLLAGEN: Provides some of the amino acids our bodies need to ma...,which is found in our nails and hair* AMINO ACIDS AND VITAMIN C: Nourish yo...,19 amino acids nourish the skins dermal layers; and Vitamin C,NaN,and formulated for healthy hair,beautiful skin,and nail support.* As we age,our collagen gets depleted,which can lead to many common signs of aging. NeoCell Super Collagen + Vita...,treat,cure,or prevent any disease; This product does not contain common GE genes or pr...,Serving Size 3 Tablets. Calories: 10; Protein: 3 g; Vitamin C (as calcium as...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2095,NaN,15d32b955eaf434c12f7b2bfded6aa80,Amazon,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Health & Household > Health Care > OTC Medications & Treatments > Digestion ...,NaN,Cluster & Eno,4.8,29,19.2,4,B000FVRR0U,NaN,NaN,https://www.amazon.com/dp/B000FVRR0U,https://m.media-amazon.com/images/I/51RZel1T8rL.jpg,44936,291924,NaN,NaN
2714,NaN,334507ac278d91d3285f8abea9cebf93,Amazon,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Health & Household > Vitamins, Minerals & Supplements > Collagen",NaN,NaN,4.3,361,120.4,28,B075LFVQGR,NaN,NaN,https://www.amazon.com/dp/B075LFVQGR,https://m.media-amazon.com/images/I/817NtyqKOGL._AC_SL1500_.jpg | https://m....,44967,374983,NaN,NaN
3178,NaN,4f45ce0a621e7674985299540c61138c,Amazon,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Health & Household > Vitamins, Minerals & Supplements > Vitamins > Vitamin B...",NaN,NaN,4.4,82,4.4,1,B07PHVNJ8Q,NaN,NaN,https://www.amazon.com/dp/B07PHVNJ8Q,NaN,44967,376334,NaN,NaN
3212,NaN,51971c9a5e4e3db09e4e84b7992a0235,Walmart,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.8,646,388.8,81,NaN,NaN,NaN,https://www.walmart.com/ip/SkinnyFit-Super-Youth-Tropical-Punch-Collagen-Pow...,https://i5.walmartimages.com/seo/SkinnyFit-Super-Youth-Tropical-Punch-Collag...,45027.6581,523009,NaN,NaN
3512,Exclude,08c152cd64476875d1685229245a23c1,Kroger,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Home > Health Care > Children's Health Care,Nature's Bounty,NaN,4.9,39,122.5,25,items_457-27272895,NaN,NaN,https://www.instacart.com/store/kroger/products/27272895,NaN,45058.43501,680613,NaN,NaN
3540,Exclude,2625bbca9e36c1735c0a9fbb2d621837,Kroger,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Home > Health Care > Digestive & Stomach Relief,Culturelle,NaN,4.09,NaN,1476.49,361,items_457-52984,NaN,NaN,https://www.instacart.com/store/kroger/products/52984,https://www.instacart.com/assets/domains/product-image/file/large_e978f2f6-e...,45058.43501,680635,NaN,NaN


## 9. Training target labels

In [34]:
available_targets = [col for col in TARGET_COLUMNS if col in train.columns]
missing_target_columns = [col for col in TARGET_COLUMNS if col not in train.columns]

print("Target columns found:", available_targets)

if missing_target_columns:
    print("Expected target columns NOT found:", missing_target_columns)

for col in available_targets:
    values = train[col].dropna().astype(str)
    unique_labels = sorted(values.unique())

    print(f"\n{'=' * 70}")
    print(col)
    print("Unique labels:", len(unique_labels))
    print(unique_labels)


Target columns found: ['Mnfr', 'Brand', 'Platform', 'Segment', 'Sub-Segment', 'TargetAgeGroup']

Mnfr
Unique labels: 4
[' aids in collagen formation* GRASS-FED COLLAGEN: Super Collagen + Vitamin C & Biotin is Keto certified', ' coffee', 'All others', 'J&J']

Brand
Unique labels: 146
[' gluten-free', ' tea', '21st Century', 'Advil', 'Airborne', 'Aleve', 'Alka Seltzer', 'Allegra', 'Amazing Nutrition', 'Amazon Basic Care', 'Ancient Nutrition', 'Andrew Lessman', 'Ark Labs', 'Astepro', 'Bayer', 'Benadryl', 'Bengay', 'Best Naturals', 'Better Not Younger', 'Biofreeze', 'Biotrue', 'Bluebonnet', 'Boiron', 'Bonafide', 'Botanic Choice', 'Bronson', 'BulkSupplements', 'CVS Health', 'Carlson', 'Carlyle', 'Centrum', 'Claritin', 'Codeage', 'Country Life', 'Culturelle', 'Designs for health', 'Dimetapp', "Doctor's Best", 'Dr. Mercola', 'Emergen-C', 'Equate', 'Estroven', 'Excedrin', 'Flonase', 'Force Factor', 'GNC', 'Gaia Herbs', 'Garden of Life', 'Generic', 'Genexa', 'HALLS', 'HAWAIIPHARM', 'HUM', 'Heal

## 10. Training class counts

In [36]:
class_counts = {}

for col in available_targets:
    counts = train[col].value_counts(dropna=False)
    class_counts[col] = counts

    print(f"\n{'=' * 70}")
    print(col)
    display(counts.rename("count").to_frame())



Mnfr


,count
Mnfr,
All others,77821
NaN,5698
J&J,1030
aids in collagen formation* GRASS-FED COLLAGEN: Super Collagen + Vitamin C & Biotin is Keto certified,2
coffee,1



Brand


,count
Brand,
UnItemised brand,54059
Other Brands,3485
NOW,1253
Nature Made,894
Swanson,820
...,...
Hers,1
tea,1
Health & Her,1



Platform


...

,count
Segment,
"Vitamins, Minerals & Supplements",49091
Digestive Health,7102
External Analgesics,6069
Internal Analgesics,4808
NaN,4789
Lifestyle CHC,4551
CCFS,4267
Allergy,2737
Other Self Care,1135



Sub-Segment


,count
Sub-Segment,
Supplements,19535
Herbal & Natural (H&N),10565
Minerals,7556
NaN,7515
Single Vitamins,4936
Cold & Heat Wraps,3818
Multivitamins,3426
Ear Care,3095
All Other Digestive Health,2141



TargetAgeGroup


,count
TargetAgeGroup,
Adult,72995
NaN,7077
Children,3718
Infant,759
Paleo friendly,1
"starch or artificial flavors; 3 tablets daily""",1
starch or artificial flavors; 3 tablets daily NeoCell Super Collagen + Vitamin C and Biotin is made with grass-fed hydrolyzed collagen,1


### Target Distribution Observations

The target-label distributions show that the classification tasks differ substantially in difficulty and data quality.

* `Mnfr` is highly imbalanced and appears to contain two legitimate dominant classes (`All others` and `J&J`) plus a few malformed values.
* `Brand` contains many legitimate brands, but is strongly dominated by `UnItemised brand`.
* `Platform` has very limited labeled data and many low-frequency classes. This is the most concerning target for supervised modeling.
* `Segment` appears to contain a small number of well-supported legitimate classes plus a few malformed values.
* `Sub-Segment` contains many meaningful classes with varying levels of support, including several rare classes.
* `TargetAgeGroup` appears to consist primarily of three legitimate classes: `Adult`, `Children`, and `Infant`.

Several low-frequency labels across the target columns appear to be shifted or malformed product text rather than legitimate classes.

These distributions should be re-evaluated after the cleaning stage before defining the final modeling taxonomy.


## 11. Additional simple data-quality checks

These checks look for:

- unnamed columns
- completely empty rows
- leading/trailing spaces in text columns
- missing training target labels


In [38]:
def basic_quality_checks(df, dataset_name):
    problems = []

    unnamed = [col for col in df.columns if str(col).startswith("Unnamed:")]
    if unnamed:
        problems.append(f"Unnamed columns found: {unnamed}")

    empty_rows = df.isna().all(axis=1).sum()
    if empty_rows:
        problems.append(f"Completely empty rows: {empty_rows}")

    text_columns = df.select_dtypes(include="object").columns
    whitespace_columns = []

    for col in text_columns:
        non_null = df[col].dropna().astype(str)
        whitespace_count = (non_null != non_null.str.strip()).sum()

        if whitespace_count > 0:
            whitespace_columns.append((col, int(whitespace_count)))

    if whitespace_columns:
        problems.append(f"Columns with leading/trailing whitespace: {whitespace_columns}")

    if not problems:
        problems.append("No obvious issues found by these basic checks.")

    print(dataset_name)
    for problem in problems:
        print("-", problem)

    return problems


train_quality_problems = basic_quality_checks(train, "TRAINING DATA")
target_quality_problems = basic_quality_checks(target, "TARGET DATA")


C:\Users\sebas\AppData\Local\Temp\ipykernel_6476\2389589426.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df.select_dtypes(include="object").columns


TRAINING DATA
- Columns with leading/trailing whitespace: [('Exclude', 3), ('JoiningKey', 4), ('Retailer', 4), ('Category', 3), ('Mnfr', 3), ('Brand', 3), ('Platform', 3), ('Segment', 3), ('Sub-Segment', 3), ('TargetAgeGroup', 3), ('Sun1', 2), ('Sun2', 2), ('Sun3', 2), ('Sun4', 2), ('Sun5', 1), ('Notes', 1), ('ProductCategory', 1), ('ProductBrand', 1), ('ProductName', 9), ('ProductRating', 430), ('ProductReviewsCount', 276), ('XRatXRev', 192), ('ReviewsCount', 121), ('Sku', 68), ('Upc', 54), ('ProductModelNumber', 23), ('ProductUrl', 8), ('ProductImageUrl', 5), ('MDM_InsertDateTime', 1), ('ProductDescription', 13), ('ProductContents', 1000)]
TARGET DATA
- Columns with leading/trailing whitespace: [('Exclude', 2), ('JoiningKey', 2), ('Retailer', 2), ('Sun1', 1), ('Sun2', 1), ('Sun3', 1), ('Sun4', 1), ('Sun5', 1), ('Notes', 1), ('ProductCategory', 1), ('ProductBrand', 1), ('ProductName', 3), ('ProductRating', 158), ('ProductReviewsCount', 103), ('XRatXRev', 73), ('ReviewsCount', 54), ('S

C:\Users\sebas\AppData\Local\Temp\ipykernel_6476\2389589426.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df.select_dtypes(include="object").columns


## 12. Write `docs/research/data_profile.md`

The report is generated from the results above so the notebook remains the reproducible source of truth.


In [41]:
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

def series_as_markdown_lines(series):
    lines = []
    for label, count in series.items():
        label_text = "<MISSING>" if pd.isna(label) else str(label)
        lines.append(f"- `{label_text}`: {int(count)}")
    return lines


report = []

report.append("# CatalogIQ Dataset Profile")
report.append("")
report.append("This report profiles the supplied training and target datasets before cleaning or modeling.")
report.append("")

report.append("## Dataset Sizes")
report.append("")
report.append(f"- Training: **{len(train):,} rows × {len(train.columns):,} columns**")
report.append(f"- Target: **{len(target):,} rows × {len(target.columns):,} columns**")
report.append("")

report.append("## Column Names")
report.append("")
report.append("### Training")
report.extend([f"- `{col}`" for col in train.columns])
report.append("")
report.append("### Target")
report.extend([f"- `{col}`" for col in target.columns])
report.append("")

report.append("## Data Types")
report.append("")
report.append("### Training")
for col, dtype in train.dtypes.items():
    report.append(f"- `{col}`: `{dtype}`")
report.append("")
report.append("### Target")
for col, dtype in target.dtypes.items():
    report.append(f"- `{col}`: `{dtype}`")
report.append("")

report.append("## Missing Values")
report.append("")
report.append("### Training")
for col, row in train_missing.iterrows():
    report.append(
        f"- `{col}`: {int(row['missing_count']):,} missing "
        f"({row['missing_percent']:.2f}%)"
    )
report.append("")
report.append("### Target")
for col, row in target_missing.iterrows():
    report.append(
        f"- `{col}`: {int(row['missing_count']):,} missing "
        f"({row['missing_percent']:.2f}%)"
    )
report.append("")

report.append("## Duplicate Rows")
report.append("")
report.append(f"- Training duplicate rows: **{int(train_duplicate_count):,}**")
report.append(f"- Target duplicate rows: **{int(target_duplicate_count):,}**")
report.append("")

report.append("## Malformed / Shifted Row Checks")
report.append("")
report.append(
    f"- Training rows with the wrong CSV field count found in the first scan: "
    f"**{len(train_bad_rows)}**"
)
report.append(
    f"- Target rows with the wrong CSV field count found in the first scan: "
    f"**{len(target_bad_rows)}**"
)
report.append(
    f"- Training rows with at least 50% missing fields: **{len(train_suspicious):,}**"
)
report.append(
    f"- Target rows with at least 50% missing fields: **{len(target_suspicious):,}**"
)
report.append("")

report.append("## Training Target Labels")
report.append("")

for col in available_targets:
    values = train[col].dropna().astype(str)
    labels = sorted(values.unique())

    report.append(f"### {col}")
    report.append("")
    report.append(f"Unique labels: **{len(labels):,}**")
    report.append("")
    report.extend([f"- `{label}`" for label in labels])
    report.append("")

report.append("## Training Target Class Counts")
report.append("")

for col in available_targets:
    report.append(f"### {col}")
    report.append("")
    report.extend(series_as_markdown_lines(class_counts[col]))
    report.append("")

report.append("## Obvious Data-Quality Problems")
report.append("")
report.append("### Training")
report.extend([f"- {problem}" for problem in train_quality_problems])
report.append("")
report.append("### Target")
report.extend([f"- {problem}" for problem in target_quality_problems])
report.append("")

if missing_target_columns:
    report.append("### Missing Expected Target Columns")
    report.append("")
    report.extend([f"- `{col}`" for col in missing_target_columns])
    report.append("")

report.append("### Missing Training Target Labels")
report.append("")
for col, count in missing_target_label_counts.items():
    report.append(f"- `{col}`: {count:,}")
report.append("")

REPORT_PATH.write_text("\n".join(report), encoding="utf-8")

print(f"Report written to: {REPORT_PATH}")


Report written to: c:\Users\sebas\PycharmProjects\Git\Critical-Materials-Recovery\docs\findings\data_profile.md
